# 04 — SHAP Explainability Analysis

Generates and explores SHAP explanations for the CreditBridge XGBoost model.

Covers:
1. **Global feature importance** via SHAP summary plot
2. **Per-applicant waterfall plot** (matching API output)
3. **Per-applicant force plot** (matching API output)
4. **SHAP beeswarm** for dataset-level insight
5. **Plain-language reason generation**

> Assumes model has been trained: run `python src/model/train.py` first.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
import joblib

from src.model.train import preprocess_features
from src.explainability.shap_explainer import ShapExplainerWrapper
from src.explainability.reason_generator import generate_plain_reasons

MODEL_PATH = '../models/xgb_v1.pkl'
PROCESSED_PATH = '../data/processed/features.parquet'

payload = joblib.load(MODEL_PATH)
base_model = payload['base_model']
feature_names = payload['feature_names']

df = pd.read_parquet(PROCESSED_PATH)
df_prep = preprocess_features(df)
X = df_prep[feature_names]

print(f'Model loaded. Features: {len(feature_names)}')
print(f'Feature matrix shape: {X.shape}')

## 1. Global SHAP Summary Plot (Dataset-Level)

In [ ]:
# Use a sample of 1000 for speed
X_sample = X.sample(1000, random_state=42)

explainer = shap.TreeExplainer(base_model)
shap_values = explainer.shap_values(X_sample)

print('Computing SHAP values...')
shap.summary_plot(
    shap_values, X_sample,
    feature_names=feature_names,
    plot_type='bar',
    max_display=18,
    show=False
)
plt.title('Mean |SHAP| Feature Importance (1,000 sample)', fontweight='bold')
plt.tight_layout()
plt.savefig('../models/shap_summary_bar.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. SHAP Beeswarm Plot

In [ ]:
shap.summary_plot(
    shap_values, X_sample,
    feature_names=feature_names,
    max_display=15,
    show=False
)
plt.title('SHAP Beeswarm — Impact on Default Probability', fontweight='bold')
plt.tight_layout()
plt.savefig('../models/shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Per-Applicant Waterfall Explanation

In [ ]:
wrapper = ShapExplainerWrapper(base_model, feature_names)

# Pick one applicant from each band for comparison
from src.model.predict import probability_to_score, score_to_band

all_probs = payload['calibrated_model'].predict_proba(X)[:, 1]
all_scores = [probability_to_score(p) for p in all_probs]
all_bands = [score_to_band(s) for s in all_scores]

# Find one Prime and one Decline applicant
df_with_bands = pd.Series(all_bands, index=X.index)
prime_idx = df_with_bands[df_with_bands == 'Prime'].index[0]
decline_idx = df_with_bands[df_with_bands == 'Decline'].index[0]

print(f'Prime applicant index : {prime_idx}')
print(f'Decline applicant index: {decline_idx}')

In [ ]:
# Waterfall for Prime applicant
x_prime = X.loc[[prime_idx]]
explanation = wrapper.explain_instance(x_prime)

print('SHAP Waterfall Data (Prime applicant):')
pd.DataFrame(explanation['waterfall_data'])[['name', 'value', 'start', 'end']].round(4)

## 4. Per-Applicant Force Plot

In [ ]:
x_decline = X.loc[[decline_idx]]
force_data = wrapper.generate_force_plot_data(x_decline)

print(f'Base value   : {force_data["base_value"]:.4f}')
print(f'Output value : {force_data["output_value"]:.4f}')
print()

print('Top positive contributors (increasing default risk):')
for f in force_data['positive_features'][:5]:
    print(f'  {f["name"]:35s}  SHAP: +{f["shap_value"]:.4f}  val={f["feature_value"]}')

print()
print('Top negative contributors (reducing default risk):')
for f in force_data['negative_features'][:5]:
    print(f'  {f["name"]:35s}  SHAP: {f["shap_value"]:.4f}  val={f["feature_value"]}')

## 5. Plain-Language Reason Generation

In [ ]:
explanation_decline = wrapper.explain_instance(x_decline)
reasons = generate_plain_reasons(explanation_decline['features_shap'], top_n=3)

print('Decline applicant — plain-language reasons:')
for i, r in enumerate(reasons, 1):
    print(f'  {i}. {r["text"]}')

## 6. SHAP Dependence Plot — Top Feature

In [ ]:
mean_abs = np.abs(shap_values).mean(axis=0)
top_feature_idx = np.argmax(mean_abs)
top_feature = feature_names[top_feature_idx]

print(f'Top SHAP feature: {top_feature}')

shap.dependence_plot(
    top_feature_idx,
    shap_values,
    X_sample,
    feature_names=feature_names,
    show=False
)
plt.title(f'SHAP Dependence: {top_feature}', fontweight='bold')
plt.tight_layout()
plt.savefig('../models/shap_dependence_top.png', dpi=150, bbox_inches='tight')
plt.show()